B1 — clone + patches + per-keypoint MD dump



In [1]:
import subprocess, re, pathlib
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
               "git clone -q https://github.com/CIawevy/FreeFine.git", shell=True, check=True)
root = pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp = root/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args, '3d')"))
for f in [root/"MD"/"mean_distance.py", root/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
md = root/"MD"/"mean_distance.py"; src = md.read_text()
m = re.search(r'^([ \t]*)all_dist = \[\]', src, re.M); ind=m.group(1)
src = src.replace(m.group(0), f"{m.group(0)}\n{ind}_PER_KP = []", 1)
assert "all_dist.append(dist)" in src
src = src.replace("all_dist.append(dist)", "all_dist.append(dist); _PER_KP.append((t_img, float(dist)))", 1)
m2 = re.search(r'^([ \t]*)md = torch\.tensor\(all_dist\)\.mean\(\)\.item\(\)', src, re.M); k=m2.group(1)
dump = (f"{m2.group(0)}\n{k}import os as _os, csv as _csv\n{k}_dp=_os.environ.get('MD_DUMP_PATH')\n"
        f"{k}if _dp:\n{k}    with open(_dp,'w',newline='') as _f:\n"
        f"{k}        _w=_csv.writer(_f); _w.writerow(['gen','dist']); _w.writerows(_PER_KP)\n"
        f"{k}    print(f'MD per-kp dumped: {{len(_PER_KP)}} rows -> {{_dp}}', flush=True)")
src = src.replace(m2.group(0), dump, 1)
md.write_text(src); print("patched: repro fixes + per-keypoint MD dump")

patched: repro fixes + per-keypoint MD dump


B2 — metric_env

In [2]:
%%bash
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
VENV=/kaggle/temp/metric_env; PY=$VENV/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf $VENV && uv venv --python 3.10.13 $VENV
uv pip install --python $PY torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python $PY "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/metric_req.txt
uv pip install --python $PY -r /tmp/metric_req.txt
uv pip install --python $PY "setuptools<70"
uv pip install --python $PY --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python $PY "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $VENV -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true; done
echo "metric_env ready -> datasets $($PY -c 'import datasets; print(datasets.__version__)')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 71.9 MB/s eta 0:00:00
metric_env ready -> datasets 2.21.0


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.47s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 700ms
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded pillow
 Downloaded networkx
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded numpy
 Downloaded triton
 Downloaded nvidia-curand-cu12
 Downloaded sympy
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 46.99s
Installed 27 packages in 347ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1

B3 — GeoBench-2D

In [3]:
import os, glob, shutil
CACHE = glob.glob("/kaggle/input/*/**/Geo-Bench-2D", recursive=True)[0].rsplit("/Geo-Bench-2D",1)[0]
GEO   = "/kaggle/temp/GeoBenchMeta"; os.makedirs(GEO, exist_ok=True)
ann = glob.glob(f"{CACHE}/**/annotation_2d.json", recursive=True)[0]
shutil.copy(ann, f"{GEO}/annotation_2d.json")
gb_dst=f"{GEO}/Geo-Bench-2D"
if os.path.islink(gb_dst): os.remove(gb_dst)
elif os.path.isdir(gb_dst): shutil.rmtree(gb_dst)
os.symlink(f"{CACHE}/Geo-Bench-2D", gb_dst)
print("GeoBench linked from cache (no HF download)", flush=True)

GeoBench linked from cache (no HF download)


B4 — symlink gen + reconstruct full manifest

In [4]:
import os, glob, json, shutil
DS ="/kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen"   # confirm
GEO="/kaggle/temp/GeoBenchMeta"
PNG_ROOT=os.path.join(DS,"gen_results_2d_final","gen_results_2d_backup")
assert os.path.isdir(PNG_ROOT), f"fix DS: {PNG_ROOT}"
link=os.path.join(GEO,"Gen_results_FreeFine_2d")
if os.path.islink(link): os.remove(link)
elif os.path.isdir(link): shutil.rmtree(link)
os.symlink(PNG_ROOT, link)
print("gen images:", len(glob.glob(f"{link}/**/*.png", recursive=True)))
man=f"{GEO}/generated_results_freefine_2d.json"
ann=json.load(open(f"{GEO}/annotation_2d.json")); added=0
for d,da in ann.items():
    for i,ins in da.get("instances",{}).items():
        for e in list(ins):
            rel=f"Gen_results_FreeFine_2d/{d}/{i}/{e}.png"
            if os.path.exists(os.path.join(GEO,rel)):
                ins[e]["gen_img_path"]=rel; added+=1
json.dump(ann,open(man,"w")); print("manifest entries:", added)   # 5677

gen images: 5677
manifest entries: 5677


B5 — 2-GPU MD shard + dump + combine



In [5]:
import os, re, json, time, select, subprocess, csv
GEO="/kaggle/temp/GeoBenchMeta"; MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"; os.environ.setdefault("HF_HOME","/kaggle/temp/hf")
from huggingface_hub import snapshot_download
snapshot_download("sd2-community/stable-diffusion-2-1", token=os.environ.get("HF_TOKEN"),
    allow_patterns=["*.json","*.txt","tokenizer/*","scheduler/*","feature_extractor/*",
                    "text_encoder/*.bin","unet/*.bin","vae/*.bin"]); print("SD-2.1 cached", flush=True)
data=json.load(open(f"{GEO}/generated_results_freefine_2d.json"))
leaves=[(d,i,c) for d,da in data.items() for i,ins in da["instances"].items() for c in ins]
mid=len(leaves)//2; subs=[set(leaves[:mid]), set(leaves[mid:])]
print(f"{len(leaves)} cases -> {len(subs[0])}/{len(subs[1])}", flush=True)
def build(s):
    o={}
    for d,da in data.items():
        ni={i:{c:v for c,v in ins.items() if (d,i,c) in s} for i,ins in da["instances"].items()}
        ni={i:k for i,k in ni.items() if k}
        if ni: nd={k:v for k,v in da.items() if k!="instances"}; nd["instances"]=ni; o[d]=nd
    return o
half=[f"/kaggle/temp/md_half_{i}.json" for i in (0,1)]; dump=[f"/kaggle/working/md_kp_{i}.csv" for i in (0,1)]
for i,s in enumerate(subs): json.dump(build(s), open(half[i],"w"))
def cmd(p): return [PY,"main.py","--path",p,"--use_relative_path","--base_dir",GEO,
    "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task","000000100","--level","0"]
base=os.environ.copy(); base.update({"PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
    "MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1","HF_HOME":"/kaggle/temp/hf"})
procs,bufs=[],["",""]
for g in (0,1):
    e=base.copy(); e["CUDA_VISIBLE_DEVICES"]=str(g); e["MD_DUMP_PATH"]=dump[g]
    procs.append(subprocess.Popen(cmd(half[g]),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,env=e,cwd=MET,bufsize=0))
print("MD launched on GPU0 + GPU1\n", flush=True)
fds={p.stdout.fileno():i for i,p in enumerate(procs)}; openf=set(fds); start=last=time.time()
def prog(b):
    m=re.findall(r"(\d+)/(\d+)\s*\[([^\]]*)\]", b[-4000:])
    if m: c,t,info=m[-1]; return f"{c}/{t} [{info}]"
    ls=[l for l in b[-2000:].replace("\r","\n").splitlines() if l.strip()]; return ls[-1][:70] if ls else "(loading...)"
while openf:
    for fd in select.select(list(openf),[],[],2.0)[0]:
        ch=os.read(fd,65536)
        if not ch: openf.discard(fd); continue
        bufs[fds[fd]]+=ch.decode("utf-8","replace")
    if time.time()-last>=30 or not openf:
        el=int(time.time()-start); print(f"[t={el//60}m{el%60:02d}s] GPU0:{prog(bufs[0])} || GPU1:{prog(bufs[1])}", flush=True); last=time.time()
for p in procs: p.wait()
rows=[]
for d in dump:
    if os.path.exists(d):
        with open(d) as f:
            r=csv.reader(f); next(r,None); rows+=[(g,float(x)) for g,x in r]
with open("/kaggle/working/md_per_kp_full.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["gen","dist"]); w.writerows(rows)
pooled=sum(x for _,x in rows)/len(rows) if rows else float("nan")
print(f"\nper-kp rows: {len(rows)} over {len(set(g for g,_ in rows))} cases")
print(f"POOLED MD (exact) = {pooled:.6f}    [reproduced/sharded was 8.4636]")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

SD-2.1 cached
5677 cases -> 2838/2839
MD launched on GPU0 + GPU1

[t=0m31s] GPU0:-----MD----- || GPU1:-----MD-----
[t=1m01s] GPU0:0/2838 [00:00<?, ?it/s] || GPU1:0/2839 [00:00<?, ?it/s]
[t=1m32s] GPU0:5/2838 [00:37<5:40:11,  7.20s/it] || GPU1:5/2839 [00:38<5:52:10,  7.46s/it]
[t=2m02s] GPU0:9/2838 [01:05<5:30:44,  7.01s/it] || GPU1:9/2839 [01:08<5:56:26,  7.56s/it]
[t=2m32s] GPU0:13/2838 [01:34<5:45:00,  7.33s/it] || GPU1:12/2839 [01:32<6:11:22,  7.88s/it]
[t=3m03s] GPU0:17/2838 [02:05<5:56:47,  7.59s/it] || GPU1:16/2839 [02:07<6:41:49,  8.54s/it]
[t=3m33s] GPU0:21/2838 [02:33<5:41:40,  7.28s/it] || GPU1:19/2839 [02:32<6:34:57,  8.40s/it]
[t=4m04s] GPU0:26/2838 [03:09<5:29:20,  7.03s/it] || GPU1:23/2839 [03:06<6:35:31,  8.43s/it]
[t=4m35s] GPU0:30/2838 [03:36<5:29:27,  7.04s/it] || GPU1:27/2839 [03:40<6:36:47,  8.47s/it]
[t=5m07s] GPU0:35/2838 [04:11<5:28:21,  7.03s/it] || GPU1:31/2839 [04:09<5:57:04,  7.63s/it]
[t=5m38s] GPU0:39/2838 [04:40<5:38:46,  7.26s/it] || GPU1:35/2839 [04:39<5